# Library

In [ ]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

# 라이브러리 설치
!pip install --upgrade --no-cache-dir numpy seaborn
!pip install ydata_profiling
!pip install missingno
!pip install tqdm

!pip install -U kss==5.2.0
!pip install kiwipiepy
!pip install soynlp
!pip install keybert
!pip install keybert[gensim]
!pip install sentence_transformers

!pip install nltk
!pip install konlpy
!pip install gensim
!pip install bertopic -U
!pip install bertopic[visualization] -U
!pip install -U accelerate
!pip install -U transformers
!pip install datasets

!pip install catboost
!pip install tensorflow==2.15 keras==2.15
!pip install keras-tqdm
!pip install shap

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 74.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.4 which is incompatible.
catboost 1.2.7 requires numpy<2.0,>=1.16.0, but you have numpy 2.2.4 which is incompatible.
ydata-profiling 4.16.1 requires numpy<2.2,>=1.16.0, but you have numpy 2.2.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.

In [ ]:
# Auto reload of library
%load_ext autoreload
%autoreload 2

# System related and data input controls
import os

# Ignore the warnings
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

# Visualization
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
## 한글 폰트 설치
!apt-get update -qq
!apt-get install fonts-nanum* -qq
## NanumGothic 폰트 경로 지정
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_prop = fm.FontProperties(fname=font_path)
## 한글 폰트 설정
matplotlib.rcParams['font.family'] = font_prop.get_name()
plt.rc('font', family='NanumGothic')
sns.set(font=font_prop.get_name())
## 마이너스 표시 설정
plt.rcParams['axes.unicode_minus'] = False

# Custom
## 사용자의 실제 작업경로로 설정!
work_path = '/content/drive/MyDrive/Research/Analysis/Lecture/특강_20250412_한국지능정보사회진흥원_빅데이터센터'
os.chdir(work_path)
!ls
from module_KK import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 BA1_TargetMarketing_DataPreprocessing_KK.ipynb     Data	   __pycache__
 BA1_TargetMarketing_ModelingBase_KK.ipynb	    mlruns	   README.md
 BA2_DemandForecasting_DataPreprocessing_KK.ipynb   Model	   Result
 BA2_DemandForecasting_DataSentiment_KK.ipynb	    module_KK.py  '나눔 글꼴'
 BA2_DemandForecasting_ModelingAI_KK.ipynb	    outputs


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


# Hyperparameter

In [3]:
# Data Preprocessing
folder_location = os.path.join(os.path.join('.', 'Data'))
Y_colname = '승차인원수'
RANDOM_STATE = 123
DATE_SPLITS = ['2023-03-31', '2024-03-31']


# Data Loading

---

1. **활용데이터 아이디어:**

 0) **비즈니스 정보:**
> 수요에 영향을 줄 수 있는 비즈니스 내부 데이터와 코로나 대휴행과 같은 과거 이벤트를 반영하여 미래에는 유무에 따른 수요를 예측할 수 있음

 1) **시계열 정보(Time Sequence):**
> 시간 정보에서 산출될 수 있는 주말, 주중, 휴일, 공휴일, 대체공휴일, 명절 등의 이벤트가 수요에 영향을 줄 수 있음

 2) **소비자심리지수(CSI):**
> 소비자의 경제 상황에 대한 심리를 지수화한 것으로, 100을 기준으로 높으면 경제에 대한 기대감이 높음을, 낮으면 반대를 의미함. 소비자심리가 개선되면 여행 및 교통수단 이용이 증가할 수 있음.

 3) **소비자물가지수(CPI):**
> 소비자가 구입하는 상품과 서비스의 가격 변동을 측정한 지표로, 물가 상승률을 파악하는 데 사용됨. 물가 상승은 실질 소득 감소로 이어져 교통비 지출에 영향을 줄 수 있음.

 4) **경제활동인구 지표:**
> **고용률:** 15세 이상 인구 중 취업자의 비율을 나타내며, 고용 안정성은 교통수단 이용 패턴에 영향을 줄 수 있음.
>
> **실업률:** 경제활동인구 중 실업자의 비율로, 실업률 상승은 소비 지출 감소로 이어져 KTX 이용에도 영향을 미칠 수 있음.

 5) **인구 및 인구이동 관련 지표:**
> **인구이동 통계:** 월별 전입 및 전출 인구 수를 통해 지역 간 이동 패턴을 파악할 수 있으며, 이는 KTX 수요 예측에 도움됨.
>
> **생활인구 통계:** 특정 지역의 시간대별 체류 인구를 파악할 수 있는 지표로, 지역별 KTX 이용 수요 분석에 활용될 수 있음.

 6) **대체 교통수단 지표:**
> **고속도로 교통량:** 월별 고속도로 이용 차량 수를 통해 도로 혼잡도를 파악하고, 이는 KTX 선호도에 영향을 줄 수 있음.
>
> **국내외 항공 여객 수:** 월별 항공 이용객 수를 통해 항공과 KTX 간의 경쟁 관계를 분석할 수 있으며 수요에 영향.

 7) **네이버 뉴스 이슈:**
> KTX, 코레일과 같은 수요에 직접적으로 검색 될 수 있는 뉴스들의 트래픽과 감성이 수요에 영향을 줄 수 있음

---

2. **데이터 소스:**

 1) [**국가통계포털(KOSIS):**](https://kosis.kr/index/index.do) 통계청에서 운영하는 포털로, 다양한 경제 지표의 월별 데이터를 제공

 2) [**한국은행 경제통계시스템(ECOS):**](https://ecos.bok.or.kr/​) 소비자물가지수, 고용률 등 다양한 경제 지표의 월별 데이터를 제공

 3) [**공공데이터포털:**](https://www.data.go.kr) 행정안전부에서 제공하는 국가·지역별 경제현황 데이터를 API 형태로 제공

 4) [**통계청 통계데이터센터(Nowcast 지표):**](https://data.kostat.go.kr) 신용카드 이용금액, 전자지급서비스 충전액 등 최신 월간 지표를 제공

 5) [**e-나라지표:**](https://www.index.go.kr/​) 외교부 제공 환율, 고용률 등 다양한 지표의 월별 데이터를 제공​

 6) [**고속도로 공공데이터 포털:**](https://data.ex.co.kr/) 한국도로공사가 제공하는 고속도로 내 유동량 데이터 제공

 7) [**빅카인즈 포털:**](https://www.bigkinds.or.kr/) 한국언론진흥재단이 제공하는 네이버 뉴스 정보와 요약 분석 제공

## 1) Business Data

## 2) ECON Data

## 3) KOSIS Data

## 4) Naver News Data

## 4') Naver Buzz & Sentiment

## 5) Data Merge

# Data Preprocessing

- 불필요 변수 삭제
- 결측치 채우기
- 이상치 처리
- 데이터 변환
- 종속변수/독립변수 & Train/Test 분리
- 스케일링

# Data Understanding

# Data Process Summary